<a href="https://colab.research.google.com/github/rishike/llm_from_scratch/blob/main/Causal_Self_Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Causal Attention, also known as masked attention is a special form of self attention.

It restricts the model to only consider previous and current inputs in a sequence, when processing any given token.

This is in contrast to the self attention mechanism which allows access to the entire input sequence at once.

When computing attention scores, the causal attention mechanism ensures that the model only factors in tokens that occur at or before the current token in the sequence.



In [1]:
import torch

In [2]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]
     ]
)

In [3]:
import torch.nn as nn

In [4]:
class SelfAttention_v1(nn.Module):

  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value

    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1)

    context_vec = attn_weights @ values

    return context_vec


In [5]:
class SelfAttention_v2(nn.Module):

  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    context_vec = attn_weights @ values

    return context_vec

In [7]:
d_in, d_out = 3, 2

In [8]:
sa_v2 = SelfAttention_v2(d_in, d_out)

In [10]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)


In [11]:
attn_weights

tensor([[0.1687, 0.1670, 0.1671, 0.1651, 0.1679, 0.1643],
        [0.1667, 0.1592, 0.1594, 0.1730, 0.1726, 0.1691],
        [0.1667, 0.1588, 0.1590, 0.1733, 0.1730, 0.1692],
        [0.1663, 0.1630, 0.1631, 0.1700, 0.1692, 0.1684],
        [0.1674, 0.1539, 0.1544, 0.1769, 0.1775, 0.1699],
        [0.1658, 0.1666, 0.1666, 0.1673, 0.1661, 0.1676]],
       grad_fn=<SoftmaxBackward0>)

In [12]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [13]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length)).bool())

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(self.mask[:num_tokens, :num_tokens], -torch.inf)
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.droput(attn_weights)
    context_vec = attn_weights @ values
    return context_vec

In [15]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1687, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1592, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1588, 0.1590, 0.0000, 0.0000, 0.0000],
        [0.1663, 0.1630, 0.1631, 0.1700, 0.0000, 0.0000],
        [0.1674, 0.1539, 0.1544, 0.1769, 0.1775, 0.0000],
        [0.1658, 0.1666, 0.1666, 0.1673, 0.1661, 0.1676]],
       grad_fn=<MulBackward0>)

Next step is to renormalize the attention weights to sum up to 1 again in each row.
We can acheive this by dividing each element in each row by the sum in each row.

In [16]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5116, 0.4884, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3441, 0.3277, 0.3282, 0.0000, 0.0000, 0.0000],
        [0.2510, 0.2461, 0.2462, 0.2567, 0.0000, 0.0000],
        [0.2017, 0.1854, 0.1860, 0.2131, 0.2138, 0.0000],
        [0.1658, 0.1666, 0.1666, 0.1673, 0.1661, 0.1676]],
       grad_fn=<DivBackward0>)

In [17]:
attn_scores

tensor([[ 0.0363,  0.0220,  0.0228,  0.0058,  0.0303, -0.0006],
        [-0.1884, -0.2539, -0.2516, -0.1362, -0.1391, -0.1680],
        [-0.1978, -0.2671, -0.2647, -0.1434, -0.1459, -0.1771],
        [-0.0975, -0.1254, -0.1245, -0.0659, -0.0727, -0.0797],
        [-0.3106, -0.4295, -0.4251, -0.2327, -0.2279, -0.2902],
        [-0.0136, -0.0071, -0.0075, -0.0013, -0.0115,  0.0016]],
       grad_fn=<MmBackward0>)

In [19]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[ 0.0363,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1884, -0.2539,    -inf,    -inf,    -inf,    -inf],
        [-0.1978, -0.2671, -0.2647,    -inf,    -inf,    -inf],
        [-0.0975, -0.1254, -0.1245, -0.0659,    -inf,    -inf],
        [-0.3106, -0.4295, -0.4251, -0.2327, -0.2279,    -inf],
        [-0.0136, -0.0071, -0.0075, -0.0013, -0.0115,  0.0016]],
       grad_fn=<MaskedFillBackward0>)

In [20]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5116, 0.4884, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3441, 0.3277, 0.3282, 0.0000, 0.0000, 0.0000],
        [0.2510, 0.2461, 0.2462, 0.2567, 0.0000, 0.0000],
        [0.2017, 0.1854, 0.1860, 0.2131, 0.2138, 0.0000],
        [0.1658, 0.1666, 0.1666, 0.1673, 0.1661, 0.1676]],
       grad_fn=<SoftmaxBackward0>)

Masking in Transformers sets scores for future tokens to a large negative value, making their influence in the softmax calculation.
The softmax function then recalculates attention weights only among the unmasked tokens.
This process ensures no information leakage from masked tokens, focusing the model solely on the intended data.

we could use the modified attention weights to compute the context vectors via context_vec = attn_weights @ values.


Masking Additional Attention weights with dropout

Dropout is a deep learning technique where randomly selected hidden layer units are ignored during training.
This prevents overfitting and improves generalization performance.

In [21]:
torch.ones(6, 6)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [23]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
dropout(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [24]:
torch.manual_seed(123)
dropout(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6882, 0.6553, 0.6565, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4922, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3708, 0.0000, 0.4262, 0.0000, 0.0000],
        [0.0000, 0.3332, 0.3331, 0.3346, 0.3322, 0.0000]],
       grad_fn=<MulBackward0>)

Implementing a compact casual attention class

In [25]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [26]:
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [31]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens,  d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_score = queries @ keys.transpose(1, 2)
    attn_score.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
    attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)
    context_vec = attn_weights @ values
    return context_vec




In [32]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
context_vecs

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

In [34]:
context_vecs.shape

torch.Size([2, 6, 2])